In [1]:
# Neural Identifier Training - 2-DOF Helicopter
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (2-DOF Helicopter)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a 2-DOF helicopter system.
    state = [ε, λ, ε_dot, λ_dot]
        ε: elevation angle (pitch) [rad]
        λ: travel angle (yaw) [rad]
        ε_dot: elevation angular velocity [rad/s]
        λ_dot: travel angular velocity [rad/s]
    u = [V_f, V_b]: voltages to front and back motors [V]
    
    Based on Quanser 2-DOF Helicopter model with simplified parameters
    """
    # Physical Parameters (typical values for Quanser 2-DOF helicopter)
    m_f = 0.65      # Mass of front propeller assembly (kg)
    m_b = 0.65      # Mass of back propeller assembly (kg)
    l_a = 0.66      # Distance from pitch axis to helicopter body (m)
    l_h = 0.177     # Distance from elevation axis to helicopter body (m)
    J_e = 0.91      # Moment of inertia about elevation axis (kg⋅m²)
    J_λ = 0.82      # Moment of inertia about travel axis (kg⋅m²)
    J_t = 0.044     # Moment of inertia of travel assembly (kg⋅m²)
    g = 9.81        # Gravity (m/s²)
    
    # Force constants (thrust per voltage)
    K_f = 0.5       # Front motor force constant (N/V)
    K_b = 0.5       # Back motor force constant (N/V)
    
    # Damping coefficients
    b_e = 0.8       # Elevation damping (N⋅m⋅s)
    b_λ = 0.8       # Travel damping (N⋅m⋅s)
    
    # State extraction
    ε, λ, ε_dot, λ_dot = state
    V_f, V_b = u
    
    # Thrust forces (saturate voltages)
    V_f_sat = np.clip(V_f, -10.0, 10.0)
    V_b_sat = np.clip(V_b, -10.0, 10.0)
    
    F_f = K_f * V_f_sat
    F_b = K_b * V_b_sat
    
    # Precompute trigonometric terms
    cos_ε = np.cos(ε)
    sin_ε = np.sin(ε)
    
    # Elevation dynamics (pitch)
    # Torque from motors
    τ_e_motor = l_a * (F_f - F_b)
    
    # Gravity torque
    τ_e_gravity = -l_h * g * (m_f + m_b) * cos_ε
    
    # Damping torque
    τ_e_damping = -b_e * ε_dot
    
    # Total elevation torque
    τ_e = τ_e_motor + τ_e_gravity + τ_e_damping
    
    # Elevation acceleration
    ε_ddot = τ_e / J_e
    
    # Travel dynamics (yaw)
    # Torque from motors (component perpendicular to elevation axis)
    τ_λ_motor = l_h * (F_f + F_b) * sin_ε
    
    # Damping torque
    τ_λ_damping = -b_λ * λ_dot
    
    # Gyroscopic coupling effect (simplified)
    τ_λ_gyro = 0.0  # Simplified - could add J_t * ε_dot * λ_dot for more realism
    
    # Total travel torque
    τ_λ = τ_λ_motor + τ_λ_damping + τ_λ_gyro
    
    # Travel acceleration
    λ_ddot = τ_λ / J_λ
    
    # State derivatives
    return np.array([ε_dot, λ_dot, ε_ddot, λ_ddot])

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-3):
    """
    RK4 integration step for better accuracy.
    """
    # RK4 for higher precision
    k1 = plant_dynamics(x_k, u_k)
    k2 = plant_dynamics(x_k + 0.5*dt*k1, u_k)
    k3 = plant_dynamics(x_k + 0.5*dt*k2, u_k)
    k4 = plant_dynamics(x_k + dt*k3, u_k)
    
    x_kp1 = x_k + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
    # Add small process noise (Laplace distribution for heavy tails)
    noise = np.random.laplace(0, process_noise_std, size=4)
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for 2-DOF Helicopter - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [ε, λ, ε_dot, λ_dot]
    u_input = [V_f, V_b]
    neuron_index: índice de la neurona (0=ε, 1=λ, 2=ε_dot, 3=λ_dot)
    """
    ε, λ, ε_dot, λ_dot = x_est
    V_f, V_b = u_input
    
    # Términos básicos sigmoidales
    s_ε = sigmoidal(ε)
    s_λ = sigmoidal(λ)
    s_ε_dot = sigmoidal(ε_dot)
    s_λ_dot = sigmoidal(λ_dot)
    
    # Control inputs
    s_V_f = sigmoidal(V_f)
    s_V_b = sigmoidal(V_b)
    s_V_diff = sigmoidal(V_f - V_b)  # Diferencia de voltajes (importante para pitch)
    s_V_sum = sigmoidal(V_f + V_b)    # Suma de voltajes (importante para yaw)
    
    # Trigonometric terms (important for helicopter dynamics)
    cos_ε = np.cos(ε)
    sin_ε = np.sin(ε)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para ε (elevation angle)
        # dε/dt = ε_dot
        return np.array([
            s_ε_dot,                   # Velocidad angular de elevación (término principal)
            s_ε_dot**2,                # Término no lineal
            s_ε * s_ε_dot,            # Acoplamiento posición-velocidad
            s_λ_dot,                   # Acoplamiento con travel
        ])
    
    elif neuron_index == 1:  # Neurona para λ (travel angle)
        # dλ/dt = λ_dot
        return np.array([
            s_λ_dot,                   # Velocidad angular de travel (término principal)
            s_λ_dot**2,                # Término no lineal
            s_λ * s_λ_dot,            # Acoplamiento posición-velocidad
            s_ε,                       # Influencia de la elevación
        ])
    
    elif neuron_index == 2:  # Neurona para ε_dot (elevation velocity)
        # dε_dot/dt = función de fuerzas de los motores, gravedad, damping
        return np.array([
            s_ε_dot,                   # Estado actual
            s_ε_dot**2,                # Damping no lineal
            s_V_diff,                  # Diferencia de voltajes (pitch control)
            sigmoidal(cos_ε),          # Término gravitacional
            s_ε_dot * s_λ_dot,        # Acoplamiento giroscópico
        ])
    
    elif neuron_index == 3:  # Neurona para λ_dot (travel velocity)
        # dλ_dot/dt = función de fuerzas de los motores, damping
        return np.array([
            s_λ_dot,                   # Estado actual
            s_λ_dot**2,                # Damping no lineal
            s_V_sum,                   # Suma de voltajes (yaw control)
            sigmoidal(sin_ε),          # Acoplamiento con elevación
            s_ε_dot * s_λ_dot,        # Acoplamiento giroscópico
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:  # ε
        return 4
    elif neuron_index == 1:  # λ
        return 4
    elif neuron_index == 2:  # ε_dot
        return 5
    elif neuron_index == 3:  # λ_dot
        return 5
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-3, R=1e-5):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            # Construir vector de características H_i (ecuación 8)
            z = construct_z_vector(x_k, u_k, i)
            H_i = z.reshape(-1, 1)
            
            # Error de identificación e_i(k) (ecuación 7)
            x_hat_i = np.dot(self.weights[i], z)
            e_i = x_kp1[i] - x_hat_i
            
            # Ganancia de Kalman K_i(k) (ecuación 6)
            S = self.R + (H_i.T @ self.P[i] @ H_i)[0, 0]
            K_i = (self.P[i] @ H_i).flatten() / S
            
            # Actualización de pesos ω_i(k+1)
            self.weights[i] = self.weights[i] + self.eta * K_i * e_i
            
            # Actualización de covarianza P_i(k+1)
            self.P[i] = self.P[i] - np.outer(K_i, H_i.flatten()) @ self.P[i] + self.Q_matrices[i]

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*2e-4 for i in range(n_neurons)]
        self.R = 1e-5
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    """
    Robust Particle Filter Trainer with NaN prevention strategies.
    Supports neuron-specific Q and R values.
    """
    def __init__(self, n_neurons, n_particles=500, Q_std=None, R_std=None):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        
        # Initialize particles with controlled variance
        self.particles = [np.random.randn(n_particles, get_z_size(i)) * 0.2 
                         for i in range(n_neurons)]
        
        # Initialize weights uniformly
        self.weights_pf = [np.ones(n_particles) / n_particles for _ in range(n_neurons)]
        
        # Tuning parameters - neuron-specific or default
        if Q_std is None:
            self.Q_std = [0.05] * n_neurons
        elif isinstance(Q_std, (list, np.ndarray)):
            assert len(Q_std) == n_neurons, f"Q_std must have length {n_neurons}"
            self.Q_std = list(Q_std)
        else:
            self.Q_std = [Q_std] * n_neurons
        
        if R_std is None:
            self.R_std = [0.05] * n_neurons
        elif isinstance(R_std, (list, np.ndarray)):
            assert len(R_std) == n_neurons, f"R_std must have length {n_neurons}"
            self.R_std = list(R_std)
        else:
            self.R_std = [R_std] * n_neurons
        
        self.regularization_std = 0.01
        
        # NaN prevention parameters
        self.min_weight = 1e-300
        self.max_log_likelihood = 100.0
        self.resample_threshold = 0.5
        
    def _normalize_weights(self, weights):
        """Safely normalize weights with NaN and underflow protection."""
        if np.any(~np.isfinite(weights)):
            print("⚠️  Warning: Non-finite weights detected, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        weights = np.maximum(weights, self.min_weight)
        weight_sum = np.sum(weights)
        if weight_sum < self.min_weight or not np.isfinite(weight_sum):
            print("⚠️  Warning: Invalid weight sum, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        return weights / weight_sum
    
    def _compute_log_likelihood(self, errors, neuron_idx):
        """Compute log-likelihood with numerical stability."""
        errors_clipped = np.clip(errors, -100, 100)
        log_likelihood = -0.5 * (errors_clipped / self.R_std[neuron_idx]) ** 2
        log_likelihood = np.clip(log_likelihood, -self.max_log_likelihood, 0)
        return log_likelihood
    
    def _resample_particles(self, neuron_idx):
        """Systematic resampling with regularization."""
        weights = self.weights_pf[neuron_idx]
        particles = self.particles[neuron_idx]
        n_weights = get_z_size(neuron_idx)
        
        weights = self._normalize_weights(weights)
        
        # Systematic resampling
        positions = (np.arange(self.n_particles) + np.random.random()) / self.n_particles
        cumulative_sum = np.cumsum(weights)
        
        indices = np.searchsorted(cumulative_sum, positions)
        indices = np.clip(indices, 0, self.n_particles - 1)
        
        self.particles[neuron_idx] = particles[indices].copy()
        
        # Add regularization jitter
        jitter = np.random.randn(self.n_particles, n_weights) * self.regularization_std
        self.particles[neuron_idx] += jitter
        
        self.weights_pf[neuron_idx] = np.ones(self.n_particles) / self.n_particles
    
    def update(self, x_kp1, x_k, u_k):
        """Update step with NaN prevention and neuron-specific Q/R."""
        for i in range(self.n_neurons):
            try:
                z = construct_z_vector(x_k, u_k, i)
                n_weights = get_z_size(i)
                
                if not np.all(np.isfinite(z)):
                    print(f"⚠️  Warning: Non-finite feature vector for neuron {i}, skipping update")
                    continue
                
                # 1. PREDICTION
                drift = np.random.randn(self.n_particles, n_weights) * self.Q_std[i]
                self.particles[i] += drift
                self.particles[i] = np.clip(self.particles[i], -100, 100)
                
                # 2. UPDATE
                preds = self.particles[i] @ z
                
                if not np.all(np.isfinite(preds)):
                    print(f"⚠️  Warning: Non-finite predictions for neuron {i}, resetting particles")
                    self.particles[i] = np.random.randn(self.n_particles, n_weights) * 0.1
                    continue
                
                errors = x_kp1[i] - preds
                log_likelihood = self._compute_log_likelihood(errors, i)
                log_likelihood -= np.max(log_likelihood)
                
                self.weights_pf[i] *= np.exp(log_likelihood)
                self.weights_pf[i] = self._normalize_weights(self.weights_pf[i])
                
                # 3. RESAMPLING
                weight_sq_sum = np.sum(self.weights_pf[i] ** 2)
                if weight_sq_sum > 0:
                    eff_N = 1.0 / weight_sq_sum
                else:
                    eff_N = 0
                
                if eff_N < self.resample_threshold * self.n_particles:
                    self._resample_particles(i)
                    
            except Exception as e:
                print(f"⚠️  Error in PF update for neuron {i}: {e}")
                self.particles[i] = np.random.randn(self.n_particles, get_z_size(i)) * 0.1
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
    
    def get_estimates(self):
        """Get weighted average of particles (with NaN protection)."""
        estimates = []
        for i in range(self.n_neurons):
            weights = self._normalize_weights(self.weights_pf[i])
            estimate = np.average(self.particles[i], axis=0, weights=weights)
            
            if not np.all(np.isfinite(estimate)):
                print(f"⚠️  Warning: Non-finite estimate for neuron {i}, using median")
                estimate = np.median(self.particles[i], axis=0)
            
            estimates.append(estimate)
        
        return estimates

# ============================================================
# 4) Error Metrics Functions
# ============================================================
def calculate_error_metrics(y_true, y_pred, metric_name="State"):
    """
    Calculate comprehensive error metrics.
    """
    errors = y_true - y_pred
    
    # Mean Absolute Error (MAE)
    mae = np.mean(np.abs(errors), axis=0)
    mae_total = np.mean(mae)
    
    # Root Mean Square Error (RMSE)
    rmse = np.sqrt(np.mean(errors**2, axis=0))
    rmse_total = np.sqrt(np.mean(rmse**2))
    
    # Normalized RMSE (NRMSE)
    ranges = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    ranges[ranges < 1e-10] = 1.0
    nrmse = rmse / ranges
    nrmse_total = np.mean(nrmse)
    
    # Maximum Absolute Error
    max_error = np.max(np.abs(errors), axis=0)
    
    # Mean Squared Error (MSE)
    mse = np.mean(errors**2, axis=0)
    mse_total = np.mean(mse)
    
    # R² Score
    ss_res = np.sum(errors**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1 - (ss_res / (ss_tot + 1e-10))
    r2_total = np.mean(r2)
    
    return {
        'MAE': mae,
        'MAE_total': mae_total,
        'RMSE': rmse,
        'RMSE_total': rmse_total,
        'NRMSE': nrmse,
        'NRMSE_total': nrmse_total,
        'MSE': mse,
        'MSE_total': mse_total,
        'Max_Error': max_error,
        'R2': r2,
        'R2_total': r2_total
    }

In [5]:
import time

# ============================================================
# 5) Simulation Main Loop - 2-DOF HELICOPTER
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # Main Simulation - PARALLEL CONFIGURATION
    # ============================================================
    n_steps = 1500
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 4  # [ε, λ, ε_dot, λ_dot]
    
    # Measurement noise parameters (realistic sensors with non-Gaussian characteristics)
    angle_noise_std = 0.01        # 0.01 rad (~0.57°) angle error
    velocity_noise_std = 0.02     # 0.02 rad/s angular velocity error
    
    # Non-Gaussian noise parameters
    t_df = 3.0                    # Degrees of freedom for Student's t-distribution
    mixture_outlier_prob = 0.05   # Probability of outlier (5%)
    outlier_scale = 5.0           # Outlier magnitude multiplier
    
    # ============================================================
    # GENERATE INITIAL WEIGHTS (UNIFORM DISTRIBUTION) - SHARED BY ALL FILTERS
    # ============================================================
    np.random.seed(7517)
    initial_weights = []
    for i in range(n_states):
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights.append(w_init.copy())
    
    print("Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # ============================================================
    # PF-RHONN Neuron-Specific Q and R values
    # ============================================================
    # Q_std: Process noise (particle diffusion) - controls exploration vs exploitation
    #        Higher values → more exploration, slower convergence, better for dynamic changes
    #        Lower values → less exploration, faster convergence, risk of particle depletion
    # R_std: Measurement noise - should match actual sensor characteristics
    #        Values based on: angle_noise_std=0.01 rad, velocity_noise_std=0.02 rad/s
    
    pf_Q_std = [
        0.3,   # Neuron 0 (ε): elevation angle - moderate dynamics, gravity effects
        0.4,   # Neuron 1 (λ): travel angle - higher dynamics due to gyroscopic coupling
        0.8,   # Neuron 2 (ε_dot): elevation velocity - high dynamics, strong nonlinearities
        0.10    # Neuron 3 (λ_dot): travel velocity - highest dynamics, coupled effects
    ]
    
    pf_R_std = [
        0.012,  # Neuron 0 (ε): angle measurement noise ~1.2× actual sensor noise
        0.012,  # Neuron 1 (λ): angle measurement noise ~1.2× actual sensor noise
        0.025,  # Neuron 2 (ε_dot): velocity measurement noise ~1.25× actual sensor noise
        0.025   # Neuron 3 (λ_dot): velocity measurement noise ~1.25× actual sensor noise
    ]
    
    n_particles_pf = 1000  # Increased for better coverage of weight space
    
    print("\n" + "="*70)
    print("PF-RHONN Neuron-Specific Parameters (Optimized for Helicopter):")
    print("="*70)
    state_names = ['ε (elevation)', 'λ (travel)', 'ε_dot (elev vel)', 'λ_dot (trav vel)']
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}):")
        print(f"    Q_std={pf_Q_std[i]:.4f} (process noise/particle diffusion)")
        print(f"    R_std={pf_R_std[i]:.4f} (measurement noise model)")
    print(f"\n  Number of particles: {n_particles_pf}")
    print(f"  Total particle-weights: {sum([get_z_size(i) for i in range(n_states)])} × {n_particles_pf}")
    
    # Initialize Trainers
    ekf = EKF_Trainer(n_states, eta=1.0, P0=1.0, Q=1e-3, R=1e-4)
    ukf = UKF_Trainer(n_states, eta=1.0, alpha=1e-2)
    pf = PF_Trainer(n_states, n_particles=n_particles_pf, Q_std=pf_Q_std, R_std=pf_R_std)
    
    # Set initial weights for all filters
    for i in range(n_states):
        ekf.weights[i] = initial_weights[i].copy()
        ukf.weights[i] = initial_weights[i].copy()
        pf.particles[i] = np.tile(initial_weights[i], (pf.n_particles, 1))
    
    print("\n✓ All filters initialized with the same RHONN weights")
    
    # Arrays for states
    x_true = np.zeros((n_steps, 4))
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Arrays for noisy measurements
    y_measured = np.zeros((n_steps, 4))
    
    # Training time measurements
    ekf_training_times = []
    ukf_training_times = []
    pf_training_times = []
    
    # Initial Conditions (helicopter near hover)
    x_true[0] = [0.1, 0.0, 0.0, 0.0]  # [ε, λ, ε_dot, λ_dot] - slight elevation
    x_est_ekf[0] = x_true[0]
    x_est_ukf[0] = x_true[0]
    x_est_pf[0] = x_true[0]
    
    # Add non-Gaussian noise to initial measurement
    initial_noise = np.array([
        np.random.standard_t(t_df) * angle_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * angle_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * velocity_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * velocity_noise_std * np.sqrt((t_df-2)/t_df)
    ])
    y_measured[0] = x_true[0] + initial_noise
    
    # Excitation Input (motor voltages)
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        # Rich excitation signal with multiple frequencies
        # V_f: front motor voltage
        V_f = 2.0 * np.sin(1.0 * t[k]) + 1.0 * np.sin(2.5 * t[k]) + 3.0
        # V_b: back motor voltage
        V_b = 1.5 * np.sin(1.5 * t[k]) + 0.8 * np.cos(3.0 * t[k]) + 2.8
        
        # Add step changes for better excitation
        if 400 < k < 450:
            V_f += 1.5
            V_b += 0.8
        elif 900 < k < 950:
            V_f -= 1.0
            V_b += 1.2
        elif 1200 < k < 1250:
            V_f += 0.8
            V_b -= 1.0
            
        u_hist[k] = [V_f, V_b]

    print("\n" + "="*70)
    print("Simulating 2-DOF Helicopter with PARALLEL CONFIGURATION...")
    print("="*70)
    print("\nEstructura de características por neurona:")
    for i in range(n_states):
        print(f"  Neurona {i} ({state_names[i]}): {get_z_size(i)} características")
    
    print("\nNoise Model (Non-Gaussian - Realistic Sensors):")
    print(f"  Process Noise: Laplace distribution (heavy tails)")
    print(f"  Measurement Noise: Student's t-distribution (df={t_df}) + Gaussian Mixture")
    print(f"    - Angles (ε, λ): ±{angle_noise_std:.4f} rad (~{np.degrees(angle_noise_std):.2f}°)")
    print(f"    - Angular velocities: ±{velocity_noise_std:.4f} rad/s")
    print(f"    - Outlier probability: {mixture_outlier_prob*100:.1f}% (scale: {outlier_scale}x)")
    print(f"\nIntegration Method: Runge-Kutta 4th Order (RK4)")
    print(f"Time step (dt): {dt} s")
    print(f"Simulation duration: {n_steps*dt:.1f} s ({n_steps} steps)")
    
    # Main simulation loop
    for k in range(n_steps - 1):
        # 1. Generate true next state
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        
        # 2. Create noisy measurement with NON-GAUSSIAN noise
        measurement_noise = np.zeros(4)
        
        for i in range(4):
            is_outlier = np.random.random() < mixture_outlier_prob
            
            if i < 2:  # Angles
                if is_outlier:
                    measurement_noise[i] = np.random.normal(0, angle_noise_std * outlier_scale)
                else:
                    t_sample = np.random.standard_t(t_df)
                    measurement_noise[i] = t_sample * angle_noise_std * np.sqrt((t_df-2)/t_df)
            else:  # Velocities
                if is_outlier:
                    measurement_noise[i] = np.random.normal(0, velocity_noise_std * outlier_scale)
                else:
                    t_sample = np.random.standard_t(t_df)
                    measurement_noise[i] = t_sample * velocity_noise_std * np.sqrt((t_df-2)/t_df)
        
        y_measured[k+1] = x_true[k+1] + measurement_noise
        
        # 3. Update filters with NOISY MEASUREMENTS
        
        # --- EKF Update & Predict ---
        start_time = time.perf_counter()
        ekf.update(y_measured[k+1], x_est_ekf[k], u_hist[k])
        ekf_time = time.perf_counter() - start_time
        ekf_training_times.append(ekf_time)
        
        for i in range(4):
            z_ekf = construct_z_vector(x_est_ekf[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z_ekf)
        
        # --- UKF Update & Predict ---
        start_time = time.perf_counter()
        ukf.update(y_measured[k+1], x_est_ukf[k], u_hist[k])
        ukf_time = time.perf_counter() - start_time
        ukf_training_times.append(ukf_time)
        
        for i in range(4):
            z_ukf = construct_z_vector(x_est_ukf[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z_ukf)
        
        # --- PF Update & Predict ---
        start_time = time.perf_counter()
        pf.update(y_measured[k+1], x_est_pf[k], u_hist[k])
        pf_time = time.perf_counter() - start_time
        pf_training_times.append(pf_time)
        
        w_pf = pf.get_estimates()
        for i in range(4):
            z_pf = construct_z_vector(x_est_pf[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z_pf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            print(f"Step {k}/{n_steps-1}")
            err_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
            err_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
            err_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
            print(f"  Current errors - EKF: {err_ekf:.4f}, UKF: {err_ukf:.4f}, PF: {err_pf:.4f}")

    # ============================================================
    # 6) Calculate Comprehensive Error Metrics
    # ============================================================
    
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE RESULTS - 2-DOF HELICOPTER WITH NON-GAUSSIAN NOISE")
    print("="*70)
    
    # Calculate error metrics
    metrics_ekf = calculate_error_metrics(x_true, x_est_ekf, "EKF-RHONN")
    metrics_ukf = calculate_error_metrics(x_true, x_est_ukf, "UKF-RHONN")
    metrics_pf = calculate_error_metrics(x_true, x_est_pf, "PF-RHONN")
    
    # Print metrics
    print("\n--- EKF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_ekf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_ekf['RMSE'][i]:.6f}, MAE={metrics_ekf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_ekf['NRMSE'][i]:.4f}, R²={metrics_ekf['R2'][i]:.4f}")
    
    print("\n--- UKF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_ukf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_ukf['RMSE'][i]:.6f}, MAE={metrics_ukf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_ukf['NRMSE'][i]:.4f}, R²={metrics_ukf['R2'][i]:.4f}")
    
    print("\n--- PF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_pf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_pf['RMSE'][i]:.6f}, MAE={metrics_pf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_pf['NRMSE'][i]:.4f}, R²={metrics_pf['R2'][i]:.4f}")
    
    # Training time analysis
    print("\n" + "="*70)
    print("⏱️  TIEMPOS DE ENTRENAMIENTO POR FILTRO")
    print("="*70)
    
    ekf_total_time = np.sum(ekf_training_times)
    ukf_total_time = np.sum(ukf_training_times)
    pf_total_time = np.sum(pf_training_times)
    
    ekf_mean_time = np.mean(ekf_training_times)
    ukf_mean_time = np.mean(ukf_training_times)
    pf_mean_time = np.mean(pf_training_times)
    
    ekf_std_time = np.std(ekf_training_times)
    ukf_std_time = np.std(ukf_training_times)
    pf_std_time = np.std(pf_training_times)
    
    print(f"\n📈 Estadísticas de Tiempos por Iteración:")
    print(f"\nEKF-RHONN:")
    print(f"  Total: {ekf_total_time:.6f} s | Media: {ekf_mean_time*1000:.4f} ms")
    print(f"  Std: {ekf_std_time*1000:.4f} ms | Min: {np.min(ekf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(ekf_training_times)*1000:.4f} ms")
    
    print(f"\nUKF-RHONN:")
    print(f"  Total: {ukf_total_time:.6f} s | Media: {ukf_mean_time*1000:.4f} ms")
    print(f"  Std: {ukf_std_time*1000:.4f} ms | Min: {np.min(ukf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(ukf_training_times)*1000:.4f} ms")
    
    print(f"\nPF-RHONN:")
    print(f"  Total: {pf_total_time:.6f} s | Media: {pf_mean_time*1000:.4f} ms")
    print(f"  Std: {pf_std_time*1000:.4f} ms | Min: {np.min(pf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(pf_training_times)*1000:.4f} ms")
    
    print(f"\n⚡ Comparación Relativa de Velocidad:")
    print(f"  EKF es {ukf_mean_time/ekf_mean_time:.2f}x más rápido que UKF")
    print(f"  EKF es {pf_mean_time/ekf_mean_time:.2f}x más rápido que PF")
    print(f"  UKF es {pf_mean_time/ukf_mean_time:.2f}x más rápido que PF")
    
    print(f"\n🎯 Eficiencia (R² / Tiempo de Entrenamiento):")
    print(f"  EKF: {metrics_ekf['R2_total']/ekf_total_time:.4f} R²/s")
    print(f"  UKF: {metrics_ukf['R2_total']/ukf_total_time:.4f} R²/s")
    print(f"  PF:  {metrics_pf['R2_total']/pf_total_time:.4f} R²/s")
    
    # Noise analysis
    print("\n" + "="*70)
    print("📉 ANÁLISIS DE RUIDO (Non-Gaussian)")
    print("="*70)
    
    meas_noise = y_measured - x_true
    print(f"\nCaracterísticas del ruido de medición:")
    for i, name in enumerate(state_names):
        noise_samples = meas_noise[:, i]
        kurtosis = np.mean((noise_samples - np.mean(noise_samples))**4) / (np.std(noise_samples)**4)
        skewness = np.mean((noise_samples - np.mean(noise_samples))**3) / (np.std(noise_samples)**3)
        print(f"\n  {name}:")
        print(f"    Std: {np.std(noise_samples):.6f}")
        print(f"    Kurtosis: {kurtosis:.4f} (Gaussian=3.0, higher=heavier tails)")
        print(f"    Skewness: {skewness:.4f} (Gaussian=0.0)")
        print(f"    Max abs: {np.max(np.abs(noise_samples)):.6f}")
    
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    rmse_dict = {'EKF-RHONN': metrics_ekf['RMSE_total'], 
                 'UKF-RHONN': metrics_ukf['RMSE_total'], 
                 'PF-RHONN': metrics_pf['RMSE_total']}
    best_filter = min(rmse_dict, key=rmse_dict.get)
    print(f"{best_filter} (RMSE total: {rmse_dict[best_filter]:.6f})")
    print("="*70)

Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(4,), min=-0.7101, max=0.3165, mean=-0.1913
  Neuron 1: shape=(4,), min=-0.6689, max=0.9859, mean=0.1275
  Neuron 2: shape=(5,), min=-0.7763, max=0.7480, mean=-0.2322
  Neuron 3: shape=(5,), min=-0.8238, max=0.3707, mean=-0.0684

PF-RHONN Neuron-Specific Parameters (Optimized for Helicopter):
  Neuron 0 (ε (elevation)):
    Q_std=0.3000 (process noise/particle diffusion)
    R_std=0.0120 (measurement noise model)
  Neuron 1 (λ (travel)):
    Q_std=0.4000 (process noise/particle diffusion)
    R_std=0.0120 (measurement noise model)
  Neuron 2 (ε_dot (elev vel)):
    Q_std=0.8000 (process noise/particle diffusion)
    R_std=0.0250 (measurement noise model)
  Neuron 3 (λ_dot (trav vel)):
    Q_std=0.1000 (process noise/particle diffusion)
    R_std=0.0250 (measurement noise model)

  Number of particles: 1000
  Total particle-weights: 18 × 1000

✓ All filters initialized with the same RHONN weights

Simulating 2-DOF Helicopte

In [6]:
# ============================================================
# 7) Visualization - 2-DOF Helicopter
# ============================================================

print("\nGenerando visualizaciones...")

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 14,
    'title_font_size': 16,
    'legend_font_size': 12,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'line_width_meas': 1.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# --- Gráfica de tiempos de entrenamiento ---
fig_times = go.Figure()

# Crear arrays de tiempo de simulación para el eje x
sim_time = t[1:]  # Excluir el primer tiempo (k=0)

# Agregar trazas de tiempo para cada filtro
fig_times.add_trace(go.Scatter(
    x=sim_time, y=ekf_training_times,
    mode='lines',
    name='EKF-RHONN',
    line=dict(color='#1f77b4', width=2),
))

fig_times.add_trace(go.Scatter(
    x=sim_time, y=ukf_training_times,
    mode='lines',
    name='UKF-RHONN',
    line=dict(color='#2ca02c', width=2),
))

fig_times.add_trace(go.Scatter(
    x=sim_time, y=pf_training_times,
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2),
))

fig_times.update_layout(
    title={
        'text': 'Tiempos de Entrenamiento por Iteración - Helicóptero 2-DOF',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo de Simulación (s)',
    yaxis_title='Tiempo de Entrenamiento (s)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        type='log',  # Escala logarítmica para mejor visualización
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_times.show()

# --- Gráfica de barras comparando tiempos medios ---
fig_times_bar = go.Figure()

filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
mean_times_ms = [ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000]

fig_times_bar.add_trace(go.Bar(
    x=filters,
    y=mean_times_ms,
    marker_color=['#1f77b4', '#2ca02c', '#d62728'],
    text=[f'{t:.3f} ms' for t in mean_times_ms],
    textposition='outside'
))

fig_times_bar.update_layout(
    title={
        'text': 'Tiempo Promedio de Entrenamiento por Iteración',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Tiempo Promedio (ms)',
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_times_bar.show()

# --- Visualización de trayectoria en espacio ε-λ (Elevation vs Travel) ---
fig_traj = go.Figure()

fig_traj.add_trace(go.Scatter(
    x=x_true[:, 0], y=x_true[:, 1],  # ε vs λ
    mode='lines',
    name='Trayectoria Real',
    line=dict(color='#000000', width=2),
))

# Puntos de inicio y final
fig_traj.add_trace(go.Scatter(
    x=[x_true[0, 0]], y=[x_true[0, 1]],
    mode='markers',
    name='Inicio',
    marker=dict(size=12, color='green', symbol='circle'),
))

fig_traj.add_trace(go.Scatter(
    x=[x_true[-1, 0]], y=[x_true[-1, 1]],
    mode='markers',
    name='Final',
    marker=dict(size=12, color='red', symbol='square'),
))

fig_traj.update_layout(
    title={
        'text': 'Trayectoria del Helicóptero en Espacio de Configuración',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Ángulo de Elevación ε (rad)',
    yaxis_title='Ángulo de Travel λ (rad)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_traj.show()

# --- Gráficas por Estado (con medidas ruidosas) ---
states_info = [
    {'idx': 0, 'var': 'ε', 'desc': 'Ángulo de Elevación (Pitch)', 'y_label': 'Ángulo ε (rad)'},
    {'idx': 1, 'var': 'λ', 'desc': 'Ángulo de Travel (Yaw)', 'y_label': 'Ángulo λ (rad)'},
    {'idx': 2, 'var': 'ε̇', 'desc': 'Velocidad Angular de Elevación', 'y_label': 'Velocidad ε̇ (rad/s)'},
    {'idx': 3, 'var': 'λ̇', 'desc': 'Velocidad Angular de Travel', 'y_label': 'Velocidad λ̇ (rad/s)'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # True state
    fig.add_trace(go.Scatter(
        x=t, y=x_true[:, i],
        mode='lines',
        name='Estado Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # Noisy measurements
    fig.add_trace(go.Scatter(
        x=t, y=y_measured[:, i],
        mode='markers',
        name='Medidas',
        marker=dict(size=2.5, color='rgba(138, 43, 226, 0.6)'),
        showlegend=True
    ))
    
    # EKF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_ekf[:, i],
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
        showlegend=True
    ))
    
    # UKF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_ukf[:, i],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
        showlegend=True
    ))
    
    # PF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]}',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# --- Gráfica de señales de control (voltajes de los motores) ---
fig_control = go.Figure()

fig_control.add_trace(go.Scatter(
    x=t, y=u_hist[:, 0],
    mode='lines',
    name='Voltaje Motor Frontal (V_f)',
    line=dict(color='#ff7f0e', width=2),
))

fig_control.add_trace(go.Scatter(
    x=t, y=u_hist[:, 1],
    mode='lines',
    name='Voltaje Motor Trasero (V_b)',
    line=dict(color='#9467bd', width=2),
))

fig_control.update_layout(
    title={
        'text': 'Señales de Control - Voltajes de los Motores',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='Voltaje (V)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_control.show()

# --- Gráfica de errores acumulados ---
fig_errors = go.Figure()

# Calculate cumulative errors over time
cumulative_error_ekf = np.zeros(n_steps)
cumulative_error_ukf = np.zeros(n_steps)
cumulative_error_pf = np.zeros(n_steps)

for k in range(1, n_steps):
    error_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
    error_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
    error_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
    
    cumulative_error_ekf[k] = cumulative_error_ekf[k-1] + error_ekf
    cumulative_error_ukf[k] = cumulative_error_ukf[k-1] + error_ukf
    cumulative_error_pf[k] = cumulative_error_pf[k-1] + error_pf

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_ekf,
    mode='lines',
    name='EKF-RHONN',
    line=dict(color='#1f77b4', width=2),
))

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_ukf,
    mode='lines',
    name='UKF-RHONN',
    line=dict(color='#2ca02c', width=2),
))

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_pf,
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2),
))

fig_errors.update_layout(
    title={
        'text': 'Error Acumulado en el Tiempo - Norma Euclidiana',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='Error Acumulado',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_errors.show()

# --- Gráfica de compensación tiempo-precision (Pareto) ---
fig_pareto = go.Figure()

fig_pareto.add_trace(go.Scatter(
    x=[ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000],
    y=[metrics_ekf['RMSE_total'], metrics_ukf['RMSE_total'], metrics_pf['RMSE_total']],
    mode='markers+text',
    text=['EKF', 'UKF', 'PF'],
    textposition='top center',
    marker=dict(
        size=15,
        color=['#1f77b4', '#2ca02c', '#d62728'],
        line=dict(width=2, color='DarkSlateGrey')
    ),
    name='Filtros'
))

fig_pareto.update_layout(
    title={
        'text': 'Compensación Tiempo-Precisión (Frente de Pareto)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo Promedio por Iteración (ms)',
    yaxis_title='Error Cuadrático Medio (RMSE)',
    xaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_pareto.show()

# --- Gráfica de comparación de métricas ---
fig_metrics = go.Figure()

metrics_names = ['RMSE', 'MAE', 'NRMSE']
ekf_vals = [metrics_ekf['RMSE_total'], metrics_ekf['MAE_total'], metrics_ekf['NRMSE_total']]
ukf_vals = [metrics_ukf['RMSE_total'], metrics_ukf['MAE_total'], metrics_ukf['NRMSE_total']]
pf_vals = [metrics_pf['RMSE_total'], metrics_pf['MAE_total'], metrics_pf['NRMSE_total']]

x = np.arange(len(metrics_names))
width = 0.25

fig_metrics.add_trace(go.Bar(
    x=x - width, y=ekf_vals, width=width,
    name='EKF-RHONN',
    marker_color='#1f77b4'
))

fig_metrics.add_trace(go.Bar(
    x=x, y=ukf_vals, width=width,
    name='UKF-RHONN',
    marker_color='#2ca02c'
))

fig_metrics.add_trace(go.Bar(
    x=x + width, y=pf_vals, width=width,
    name='PF-RHONN',
    marker_color='#d62728'
))

fig_metrics.update_layout(
    title={
        'text': 'Comparación de Métricas de Error',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis=dict(
        tickmode='array',
        tickvals=x,
        ticktext=metrics_names,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Valor de la Métrica',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60),
    barmode='group'
)

fig_metrics.show()

print("\n✅ Visualización completa para Helicóptero 2-DOF con configuración paralela.")

# ============================================================
# 8) Additional Analysis
# ============================================================

print("\n" + "="*70)
print("📈 ANÁLISIS ADICIONAL")
print("="*70)

# Analyze convergence
convergence_window = 100  # Look at last 100 steps
if n_steps > convergence_window:
    final_metrics = {
        'EKF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                       x_est_ekf[-convergence_window:, :]),
        'UKF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                       x_est_ukf[-convergence_window:, :]),
        'PF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                      x_est_pf[-convergence_window:, :])
    }
    
    print(f"\nMétricas en los últimos {convergence_window} pasos (estado estacionario):")
    for filter_name, metrics in final_metrics.items():
        print(f"\n{filter_name}:")
        print(f"  RMSE total: {metrics['RMSE_total']:.6f}")
        for i, name in enumerate(state_names):
            print(f"    {name}: RMSE={metrics['RMSE'][i]:.6f}, R²={metrics['R2'][i]:.4f}")

# Resumen final
print("\n" + "="*70)
print("🎯 RESUMEN EJECUTIVO - HELICÓPTERO 2-DOF")
print("="*70)
print(f"Filtro más preciso (menor RMSE): {best_filter}")
print(f"Filtro más rápido: {'EKF-RHONN' if ekf_mean_time == min([ekf_mean_time, ukf_mean_time, pf_mean_time]) else 'UKF-RHONN' if ukf_mean_time == min([ekf_mean_time, ukf_mean_time, pf_mean_time]) else 'PF-RHONN'}")
print(f"\nRecomendaciones:")
print("1. Para control en tiempo real: EKF-RHONN (velocidad)")
print(f"2. Para máxima precisión: {best_filter}")
print("3. Para robustez frente a ruido no-Gaussiano: PF-RHONN")
print("\nIntegration Method Used: Runge-Kutta 4th Order (RK4) - High accuracy discretization")
print("Error Metrics: RMSE, MAE, NRMSE, R², Max Error")
print("\nSistema: Helicóptero 2-DOF (Quanser-based model)")
print("Estados: Elevación (ε), Travel (λ), Velocidades angulares")


Generando visualizaciones...



✅ Visualización completa para Helicóptero 2-DOF con configuración paralela.

📈 ANÁLISIS ADICIONAL

Métricas en los últimos 100 pasos (estado estacionario):

EKF:
  RMSE total: 0.022194
    ε (elevation): RMSE=0.009202, R²=0.9749
    λ (travel): RMSE=0.020977, R²=0.9916
    ε_dot (elev vel): RMSE=0.028172, R²=0.1531
    λ_dot (trav vel): RMSE=0.025534, R²=-1.1592

UKF:
  RMSE total: 0.048318
    ε (elevation): RMSE=0.010196, R²=0.9692
    λ (travel): RMSE=0.093661, R²=0.8326
    ε_dot (elev vel): RMSE=0.019070, R²=0.6120
    λ_dot (trav vel): RMSE=0.009925, R²=0.6738

PF:
  RMSE total: 0.022506
    ε (elevation): RMSE=0.010743, R²=0.9659
    λ (travel): RMSE=0.017099, R²=0.9944
    ε_dot (elev vel): RMSE=0.030274, R²=0.0220
    λ_dot (trav vel): RMSE=0.026491, R²=-1.3241

🎯 RESUMEN EJECUTIVO - HELICÓPTERO 2-DOF
Filtro más preciso (menor RMSE): EKF-RHONN
Filtro más rápido: EKF-RHONN

Recomendaciones:
1. Para control en tiempo real: EKF-RHONN (velocidad)
2. Para máxima precisión: EKF-RHO

In [7]:
# Imprimir pesos finales de cada filtro
print("\n" + "="*70)
print("🔍 PESOS FINALES DE LAS REDES NEURONALES")
print("="*70)

print("\n--- EKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ekf.weights[i]}")
    print(f"  Norm: {np.linalg.norm(ekf.weights[i]):.4f}, Mean: {np.mean(ekf.weights[i]):.4f}, Std: {np.std(ekf.weights[i]):.4f}")

print("\n--- UKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ukf.weights[i]}")
    print(f"  Norm: {np.linalg.norm(ukf.weights[i]):.4f}, Mean: {np.mean(ukf.weights[i]):.4f}, Std: {np.std(ukf.weights[i]):.4f}")

print("\n--- PF-RHONN ---")
w_pf_final = pf.get_estimates()
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {w_pf_final[i]}")
    print(f"  Norm: {np.linalg.norm(w_pf_final[i]):.4f}, Mean: {np.mean(w_pf_final[i]):.4f}, Std: {np.std(w_pf_final[i]):.4f}")

print("\n" + "="*70)
print("✅ SIMULACIÓN COMPLETADA - HELICÓPTERO 2-DOF")
print("="*70)
print("\nSistema: Helicóptero 2-DOF (Quanser-based model)")
print(f"Método de discretización: Runge-Kutta 4° Orden (RK4)")
print(f"Paso de tiempo: {dt} s")
print(f"Duración: {n_steps*dt:.1f} s")
print(f"Número de estados: {n_states}")
print("\nEstados:")
print("  - ε: Ángulo de elevación (pitch)")
print("  - λ: Ángulo de travel (yaw)")
print("  - ε̇: Velocidad angular de elevación")
print("  - λ̇: Velocidad angular de travel")
print("\nEntradas de control:")
print("  - V_f: Voltaje motor frontal")
print("  - V_b: Voltaje motor trasero")
print("\nMétricas de error utilizadas:")
print("  - RMSE (Root Mean Square Error)")
print("  - MAE (Mean Absolute Error)")
print("  - NRMSE (Normalized Root Mean Square Error)")
print("  - R² (Coefficient of Determination)")
print("  - Max Error (Maximum Absolute Error)")



🔍 PESOS FINALES DE LAS REDES NEURONALES

--- EKF-RHONN ---

Neurona 0 (ε (elevation)) - 4 pesos:
  [-5.04669056  5.25525628 -2.22761962 -0.49121035]
  Norm: 7.6348, Mean: -0.6276, Std: 3.7655

Neurona 1 (λ (travel)) - 4 pesos:
  [-27.9677246   34.98361049  -2.52324349 -13.8510674 ]
  Norm: 46.9496, Mean: -2.3396, Std: 23.3579

Neurona 2 (ε_dot (elev vel)) - 5 pesos:
  [ 1.19987206 -3.50960525 -0.02362809 -0.01538016  0.67994352]
  Norm: 3.7710, Mean: -0.3338, Std: 1.6531

Neurona 3 (λ_dot (trav vel)) - 5 pesos:
  [-0.81353415 -1.96402527 -0.34722481  0.23850651 -0.82272019]
  Norm: 2.3181, Mean: -0.7418, Std: 0.7242

--- UKF-RHONN ---

Neurona 0 (ε (elevation)) - 4 pesos:
  [-4.97005729  3.48706085 10.46296282 -2.77149076]
  Norm: 12.4103, Mean: 1.5521, Std: 6.0079

Neurona 1 (λ (travel)) - 4 pesos:
  [-29.36204828  23.63557815  16.15120665  -5.10531806]
  Norm: 41.3243, Mean: 1.3299, Std: 20.6193

Neurona 2 (ε_dot (elev vel)) - 5 pesos:
  [-0.74112989  2.27959321  0.0951287  -1.57812